## Section 1 — Dataset Acquisition

Load the Oxford-IIIT Pet dataset, inspect its class distribution and image resolutions, and build a fixed, stratified train/val split.

This is the **only** notebook that downloads the dataset — every later notebook opens it with `download=False`. It is also where the train/val split is decided once and persisted to `splits/train_val_indices.json`, so that:

- the synthetic images generated in Section 4 are derived strictly from **training** images, and never leak into validation;
- the baseline and augmented runs in Section 5 are trained on the same real images and validated on the same held-out ones, making their comparison fair.

The official `test` split is loaded here only to characterise it. It is kept untouched as the final held-out set — real images only, seen for the first time in Section 6.

In [1]:
import random

import numpy as np
import torch


def set_global_seed(seed: int) -> None:
    """Seed Python, NumPy and PyTorch RNGs for reproducible runs.

    Args:
        seed: the seed value applied to all random number generators.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


SEED = 42

set_global_seed(SEED)

print(f"Seeded every RNG with {SEED}")

Seeded every RNG with 42


### 1.1 Loading the dataset

`torchvision` exposes only `trainval` and `test` splits for Oxford-IIIT Pet, with `target_types="category"` giving breed-level labels (37 classes) rather than the binary cat/dog species label.

In [2]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from torchvision.datasets import OxfordIIITPet

DATA_ROOT = Path("data")
SPLITS_DIR = Path("splits")
TRAIN_VAL_SPLIT_PATH = SPLITS_DIR / "train_val_indices.json"
VAL_SIZE = 0.2

DATA_ROOT.mkdir(exist_ok=True)
SPLITS_DIR.mkdir(exist_ok=True)


def load_oxford_pet_split(root: Path, split: str, download: bool = True) -> OxfordIIITPet:
    """Load one official split of the Oxford-IIIT Pet dataset with breed-level labels.

    Args:
        root: directory where the dataset is (or will be) cached on disk.
        split: torchvision split name, either "trainval" or "test".
        download: whether to download the dataset if not already present in `root`.

    Returns:
        OxfordIIITPet: dataset yielding (PIL.Image, breed_label) pairs.
    """
    return OxfordIIITPet(root=str(root), split=split, target_types="category", download=download)


def get_labels(dataset: OxfordIIITPet) -> np.ndarray:
    """Extract the integer breed label for every sample in an OxfordIIITPet dataset.

    Uses the dataset's internal `_labels` cache when available (fast path) and
    falls back to iterating the dataset otherwise.

    Args:
        dataset: an OxfordIIITPet dataset instance.

    Returns:
        np.ndarray: integer class label for each sample, in dataset order.
    """
    labels = getattr(dataset, "_labels", None)
    if labels is not None:
        return np.asarray(labels)
    return np.asarray([label for _, label in dataset])

In [3]:
# The only `download=True` in the pipeline: notebooks 02-06 all open the dataset with
# `download=False` and rely on this cell having fetched it (~1.6 GB in `data/` once extracted).
trainval_dataset = load_oxford_pet_split(DATA_ROOT, split="trainval", download=True)
test_dataset = load_oxford_pet_split(DATA_ROOT, split="test", download=True)
class_names = trainval_dataset.classes

trainval_labels = get_labels(trainval_dataset)
test_labels = get_labels(test_dataset)

print(f"Trainval images: {len(trainval_dataset)} | Test images: {len(test_dataset)} | Classes: {len(class_names)}")

Trainval images: 3680 | Test images: 3669 | Classes: 37


### 1.2 Inspecting the dataset

Two properties decide how the later stages are built:

- **Class balance** — Oxford-IIIT Pet is close to uniform (~100 images per breed). That is why Section 6 reports macro averages alongside weighted ones, and why a per-class synthetic-image budget is a reasonable augmentation strategy.
- **Resolution spread** — images vary widely in size, so every downstream stage resizes explicitly rather than assuming a native resolution.

In [4]:
def class_distribution(labels: np.ndarray, class_names: list[str]) -> pd.DataFrame:
    """Count samples per class label.

    Args:
        labels: integer class label for each sample.
        class_names: breed name for each class index.

    Returns:
        pd.DataFrame: one row per class with its name and sample count, sorted by count.
    """
    counts = Counter(labels.tolist())
    rows = [
        {"class_id": idx, "class_name": name, "count": counts.get(idx, 0)}
        for idx, name in enumerate(class_names)
    ]
    return pd.DataFrame(rows).sort_values("count").reset_index(drop=True)


def sample_image_resolutions(dataset: OxfordIIITPet, sample_size: int, seed: int) -> pd.DataFrame:
    """Sample image resolutions to characterize the dataset's size variability.

    Args:
        dataset: an OxfordIIITPet dataset instance (no transform applied, so `__getitem__`
            returns PIL images with their native size).
        sample_size: number of images to sample without replacement.
        seed: seed controlling which indices are sampled.

    Returns:
        pd.DataFrame: width/height (in pixels) for each sampled image.
    """
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset), size=min(sample_size, len(dataset)), replace=False)
    rows = [{"width": dataset[i][0].width, "height": dataset[i][0].height} for i in indices]
    return pd.DataFrame(rows)


trainval_distribution = class_distribution(trainval_labels, class_names)
test_distribution = class_distribution(test_labels, class_names)

print("Trainval per-class count (min/max/mean):")
print(trainval_distribution["count"].agg(["min", "max", "mean"]))
trainval_distribution

Trainval per-class count (min/max/mean):
min      93.000000
max     100.000000
mean     99.459459
Name: count, dtype: float64


,class_id,class_name,count
0,11,Egyptian Mau,93
1,22,Newfoundland,96
2,12,English Cocker Spaniel,96
3,7,Bombay,96
4,32,Siamese,99
5,0,Abyssinian,100
6,23,Persian,100
7,24,Pomeranian,100
8,25,Pug,100
9,26,Ragdoll,100


In [5]:
resolution_sample = sample_image_resolutions(trainval_dataset, sample_size=200, seed=SEED)
resolution_sample.describe()

,width,height
count,200.000000,200.000000
mean,448.740000,400.180000
std,224.934167,173.824863
min,199.000000,182.000000
25%,349.500000,333.000000
50%,500.000000,375.000000
75%,500.000000,500.000000
max,3264.000000,2448.000000


### 1.3 Train / validation split

The `trainval` split is divided once, stratified by breed so every class keeps its proportion on both sides, and the indices are written to `splits/train_val_indices.json`.

Persisting it is what makes the whole pipeline reproducible across separate notebook sessions: notebooks 02, 04 and 05 each run in their own kernel and all read this one file, so they cannot silently disagree about which images are training images. `get_or_create_train_val_split` reuses the file whenever it exists and refuses to reuse one whose size no longer matches the dataset, rather than producing a split that quietly mismatches the captions and synthetic images already generated against the old one.

In [6]:
def stratified_train_val_split(labels: np.ndarray, val_size: float, seed: int) -> tuple[np.ndarray, np.ndarray]:
    """Split sample indices into stratified train/validation subsets.

    Args:
        labels: integer class label for each sample, used to preserve class balance.
        val_size: fraction of samples assigned to the validation subset.
        seed: seed controlling the split.

    Returns:
        tuple[np.ndarray, np.ndarray]: (train_indices, val_indices) into the original array.
    """
    all_indices = np.arange(len(labels))
    train_idx, val_idx = train_test_split(
        all_indices, test_size=val_size, stratify=labels, random_state=seed
    )
    return train_idx, val_idx


def get_or_create_train_val_split(path: Path, labels: np.ndarray, val_size: float, seed: int) -> tuple[np.ndarray, np.ndarray]:
    """Reuse the stratified train/val split from disk if one exists, otherwise create and save it.

    Args:
        path: destination/source JSON file storing the split indices.
        labels: integer class label for each sample, used to preserve class balance.
        val_size: fraction of samples assigned to the validation subset.
        seed: seed controlling the split.

    Returns:
        tuple[np.ndarray, np.ndarray]: (train_indices, val_indices).

    Raises:
        ValueError: if the stored split no longer covers the current dataset size.
    """
    if path.exists():
        split = json.loads(path.read_text())
        train_idx, val_idx = np.asarray(split["train_idx"]), np.asarray(split["val_idx"])
        if len(train_idx) + len(val_idx) == len(labels):
            print(f"Loaded existing train/val split from {path} (train={len(train_idx)}, val={len(val_idx)})")
            return train_idx, val_idx
        raise ValueError(f"{path} does not match the current dataset size; delete it to regenerate.")

    train_idx, val_idx = stratified_train_val_split(labels, val_size, seed)
    path.write_text(json.dumps({"train_idx": train_idx.tolist(), "val_idx": val_idx.tolist()}))
    print(f"Created new train/val split and saved it to {path} (train={len(train_idx)}, val={len(val_idx)})")
    return train_idx, val_idx


train_idx, val_idx = get_or_create_train_val_split(TRAIN_VAL_SPLIT_PATH, trainval_labels, val_size=VAL_SIZE, seed=SEED)
print(f"Train: {len(train_idx)} | Val: {len(val_idx)}")

Loaded existing train/val split from splits/train_val_indices.json (train=2944, val=736)
Train: 2944 | Val: 736


### 1.4 Handoff

Everything downstream needs is now on disk. The split indices are the only *new* artifact — the dataset itself is the second, and both are reconstructed by every later notebook from `data/` plus this one JSON file:

| Consumer | Reads |
|---|---|
| `02_Captioning.ipynb` | dataset (`trainval`) + `train_idx` — captions only training images |
| `04_ImageGeneration.ipynb` | dataset (`trainval`) + `train_idx` — generates from training images only |
| `05_ClassifierTraining.ipynb` | dataset (`trainval` + `test`) + `train_idx` / `val_idx` |
| `06_Evaluation&Comparison.ipynb` | dataset (`test`) |

In [7]:
print("Section 1 complete — artifacts on disk:")
print(f"  {DATA_ROOT}/ : Oxford-IIIT Pet, {len(trainval_dataset)} trainval + {len(test_dataset)} test images")
print(f"  {TRAIN_VAL_SPLIT_PATH} : train={len(train_idx)} | val={len(val_idx)} (stratified, seed={SEED})")
print("\nNext: run 02_Captioning.ipynb.")

Section 1 complete — artifacts on disk:
  data/ : Oxford-IIIT Pet, 3680 trainval + 3669 test images
  splits/train_val_indices.json : train=2944 | val=736 (stratified, seed=42)

Next: run 02_Captioning.ipynb.
